#### Library imports

In [14]:
import torch
import torchvision
import torch.nn as nn
import numpy as np
import wandb
import random
import sys
import scipy
from tqdm.notebook import tqdm

from typing import List, Tuple, Callable, Optional

from pathlib import Path

from torchvision.transforms import v2
import torchinfo
from kaggle_secrets import UserSecretsClient

from PIL import Image
from torch.utils.data import Dataset, DataLoader

sys.path.append("/kaggle/input/datasets/alessandrotirelli/prw-utility-functions")
from utils import parse_annotations_file, crop_detections_from_frame, set_requires_grad_for_layer

#### Parameters definition

In [15]:
do_train = False

num_workers = 4
log_interval = 5

seed = 21 # random seed

emb_size = 2048
num_instances = 8
num_ids_per_batch = 16
batch_size = num_ids_per_batch * num_instances
num_epochs = 300
lr = 1e-3
margin = 0.5 # cosine dis-similarity ranges from 0 to 2 (decreased margin to fit the range)

detection_size = (128, 256) # widely used measures in the literature

run_name = f"resnet50_bnneck_pk{num_ids_per_batch}x{num_instances}_emb{emb_size}_ep{num_epochs}"

wandb_project = "Second-Stage-PWR-Embedder"
data_root = Path("/kaggle/input/datasets/edoardomerli/prw-person-re-identification-in-the-wild") 
root_ckpts = Path("/kaggle/working/ckpts")

#### GPU Initialization

In [16]:
device = "cpu"
if torch.cuda.is_available():
  print("All good, a Gpu is available.")
  device = torch.device("cuda:0")
else:
  print("Please set GPU via Edit -> Notebook Settings.")

All good, a Gpu is available.


#### Reproducibility and Deterministic Mode

In [17]:
def fix_random(seed: int) -> None:
    """Fix all the possible sources of randomness.

    Args:
        seed: the seed to use.
    """
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

fix_random(seed=seed)

## Dataset and Dataloader

In [18]:
class DetectionDataset(Dataset):
    def __init__(
        self,
        data_root: Path,
        split_str: str = "train",
        transforms: Optional[Callable] = None
    ) -> None:

        if split_str != "train":
            raise AssertionError(
                f"{split_str} is not supported in this training-only notebook. Please use 'train'."
            )

        # DetectionDataset stores 2 lists of fixed dimensionality
        self.detections: List[Image.Image] = []
        self.pids: List[int] = []
        self.split_str = split_str
        self.transforms = transforms

        ext_images = "jpg"
        ext_annotations = "jpg.mat"
        path_images = data_root / "frames"
        path_annotations = data_root / "annotations"

        variable_name_split = "img_index_train"
        path_split = data_root / "frame_train.mat"

        # read Matlab file
        mat = scipy.io.loadmat(path_split)
        if variable_name_split not in mat:
            raise AssertionError(
                f"Missing {variable_name_split} in {path_split}."
            )
        sample_indexes_list = [str(elem[0][0]) for elem in mat[variable_name_split]] # in the string format: 'cX_sY_ZZZZZZ'

        # converting Python list to Python set to scale down access cost from O(n) to O(1) in list comprehension
        sample_indexes = set(sample_indexes_list)

        images = sorted([
            path for path in path_images.rglob(f"*.{ext_images}")
            if path.name.removesuffix(f".{ext_images}") in sample_indexes
        ])
        annotations = sorted([
            path for path in path_annotations.rglob(f"*.{ext_annotations}")
            if path.name.removesuffix(f".{ext_annotations}") in sample_indexes
        ])

        if len(images) != len(annotations):
            raise AssertionError(
                f"Images and labels differs in size: {len(images)} vs {len(annotations)}."
            )

        for image_path, annotation_path in tqdm(zip(images, annotations), total=len(images), desc="Training detections extraction"):
            ids, boxes = parse_annotations_file(annotation_path)
            boxes = [[box[0], box[1], box[0] + box[2], box[1] + box[3]] for box in boxes] # convert from whxy to xyxy box format
            image = Image.open(image_path).convert("RGB")
            detections = crop_detections_from_frame(image, boxes, detection_size)

            for pid, det in zip(ids, detections):
                pid = int(pid)
                self.detections.append(det)
                self.pids.append(pid)

    def __getitem__(self, idx):
        detection = self.detections[idx]
        if self.transforms is not None:
            detection = self.transforms(detection)

        pid = self.pids[idx]
        return detection, pid

    def __len__(self):
        return len(self.detections)

In [19]:
# Create dataset
train_transform = v2.Compose([
    v2.ToImage(), 
    v2.ToDtype(torch.float32, scale=True),
    v2.RandomApply([
        v2.GaussianBlur(
            kernel_size=(3, 9),   # randomly picks ODD kernel from this range
            sigma=(0.5, 3.0)      # randomly picks (using uniform sampling) sigma in this range
        )
    ], p=0.5), # added gaussian blur to improve generalization over scale
    #v2.RandomErasing(p=0.25, scale=(0.02, 0.1), ratio=(0.3, 3.3), value="random")
])

train_dataset = DetectionDataset(
    data_root=data_root,
    split_str="train",
    transforms=train_transform
)
print(f"Number of training samples: {len(train_dataset)}")

Training detections extraction:   0%|          | 0/5704 [00:00<?, ?it/s]

Number of training samples: 18031


In [20]:
def collate_skip_none(batch):
    batch = [item for item in batch if item is not None]
    if len(batch) == 0:
        return None
    return torch.utils.data.dataloader.default_collate(batch)

class PKBatchSampler(torch.utils.data.Sampler):
    """Samples P identities and K instances per identity for each batch."""
    def __init__(
            self, 
            labels: List[int], 
            num_ids_per_batch: int, 
            num_instances: int
    ) -> None:
        self.index_dict = {}
        for idx, pid in enumerate(labels):
            pid = int(pid)
            if pid < 0:
                continue
            self.index_dict.setdefault(pid, []).append(idx)
        self.pids = list(self.index_dict.keys())
        self.num_ids_per_batch = num_ids_per_batch
        self.num_instances = num_instances
        self.batch_size = num_ids_per_batch * num_instances

    def __iter__(self):
        pid_list = self.pids.copy()
        random.shuffle(pid_list)
        batch = []
        for pid in pid_list:
            idxs = self.index_dict[pid]
            if len(idxs) >= self.num_instances:
                sampled = random.sample(idxs, self.num_instances)
            else:
                sampled = list(np.random.choice(idxs, self.num_instances, replace=True))
            batch.extend(sampled)
            if len(batch) == self.batch_size:
                yield batch
                batch = []

    def __len__(self) -> int:
        return len(self.pids) // self.num_ids_per_batch


train_batch_sampler = PKBatchSampler(
    labels=train_dataset.pids,
    num_ids_per_batch=num_ids_per_batch,
    num_instances=num_instances
)
train_dataloader = DataLoader(
    train_dataset,
    batch_sampler=train_batch_sampler,
    pin_memory=True,
    num_workers=num_workers, 
    collate_fn=collate_skip_none
)

## Embedder Model Definition

In [21]:
class Embedder(nn.Module):
    """
        Neural network computing an image embedding.

        emb_size: the size for the embedding.
        normalize_emb: if True, normalizes the embedding using the L2 norm at eval time.
        pretrained_feat_ext: if True, uses a pretrained feature extractor.
        train_feat_ext: if True, trains the feature extractor.
    """
    def __init__(self,
                 emb_size: int,
                 normalize_emb: bool,
                 pretrained_feat_ext: bool,
                 train_feat_ext: bool
        ) -> None:
        
        super().__init__()
        
        self.backbone = torchvision.models.resnet50( # using resnet50 as widely adopted in the literature
            weights="IMAGENET1K_V2" if pretrained_feat_ext else None
        )
        feat_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        set_requires_grad_for_layer(self.backbone, train_feat_ext)

        self.emb = torch.nn.Linear(feat_dim, emb_size, bias=False)
        self.bnneck = torch.nn.BatchNorm1d(emb_size)
        self.bnneck.bias.requires_grad_(False)

        self.normalize_emb = normalize_emb

    def forward(self, x):
        feat = self.backbone(x)
        emb = self.emb(feat)
        bn_feat = self.bnneck(emb)

        if self.normalize_emb: # normalizing outputs
            bn_feat = torch.nn.functional.normalize(bn_feat)

        return bn_feat

In [22]:
embedder = Embedder(
    emb_size=emb_size,
    normalize_emb=True,
    pretrained_feat_ext=True,
    train_feat_ext=True
).to(device)

torchinfo.summary(
    embedder,
    input_size=(1, 3, detection_size[1], detection_size[0]),
    mode="eval"
)

Layer (type:depth-idx)                        Output Shape              Param #
Embedder                                      [1, 2048]                 --
├─ResNet: 1-1                                 [1, 2048]                 --
│    └─Conv2d: 2-1                            [1, 64, 128, 64]          9,408
│    └─BatchNorm2d: 2-2                       [1, 64, 128, 64]          128
│    └─ReLU: 2-3                              [1, 64, 128, 64]          --
│    └─MaxPool2d: 2-4                         [1, 64, 64, 32]           --
│    └─Sequential: 2-5                        [1, 256, 64, 32]          --
│    │    └─Bottleneck: 3-1                   [1, 256, 64, 32]          75,008
│    │    └─Bottleneck: 3-2                   [1, 256, 64, 32]          70,400
│    │    └─Bottleneck: 3-3                   [1, 256, 64, 32]          70,400
│    └─Sequential: 2-6                        [1, 512, 32, 16]          --
│    │    └─Bottleneck: 3-4                   [1, 512, 32, 16]          379,392

### Utility function: batch-hard triplet loss

The **Triplet Loss**, implemented in the next code section, given an *anchor* $A$, a *positive* $P$ and a *negative* $N$ is given by the formula: $\mathcal{L}=\max\{0, d(A,P) - d(A,N) + m\}$, where the margin $m$ is an hyperparameter used to control the separation between differently labelled groups of data.

In [23]:
def batch_hard_triplet_loss(
    embeddings: torch.Tensor,
    ids: torch.Tensor,
    margin: float
) -> Tuple[torch.Tensor, int]:
    """Computes the semi-hard triplet loss for a batch of embeddings."""
    if embeddings.size(dim=0) < 2: # corner case: 
        return embeddings.sum() * 0.0, 0

    labels = ids.view(-1)
    dist = 1.0 - embeddings @ embeddings.t() # cosine (dis)similarity
    diag = torch.eye(labels.size(0), device=labels.device, dtype=torch.bool)

    same_label = labels.unsqueeze(dim=0) == labels.unsqueeze(dim=1)
    same_label = same_label & ~diag # removing the diagonal to prevent positive the sample to coincide with the anchor samplee

    if not torch.any(same_label):
        return embeddings.sum() * 0.0, 0

    dist_pos = dist.clone()
    dist_pos[~same_label] = -float("inf")
    hardest_pos, _ = dist_pos.max(dim=1)

    valid_pos = hardest_pos != -float("inf")
    if not torch.any(valid_pos):
        return embeddings.sum() * 0.0, 0

    dist_neg = dist.clone()
    dist_neg[same_label | diag] = float("inf")

    semi_mask = (
        valid_pos.unsqueeze(1)
        & (dist_neg > hardest_pos.unsqueeze(1))
        & (dist_neg < (hardest_pos.unsqueeze(1) + margin))
    )
    semi_neg = dist_neg.clone()
    semi_neg[~semi_mask] = float("inf")
    semi_hard_neg, _ = semi_neg.min(dim=1)

    valid = semi_hard_neg != float("inf")
    if not torch.any(valid):
        return embeddings.sum() * 0.0, 0

    loss = torch.relu(hardest_pos[valid] - semi_hard_neg[valid] + margin)
    return loss.mean(), int(valid.sum().item())

 ## Training Loop

In [24]:
def train_epoch(
    model: nn.Module,
    train_dataloader: torch.utils.data.DataLoader,
    device: torch.device,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler,
    epoch: int
) -> float:
    """Trains a neural network for one epoch."""
    loss_train = 0.0
    num_batches = 0

    model.train()
    for idx_batch, batch in enumerate(train_dataloader):
        if batch is None:
            continue

        crops, ids = batch
        crops = crops.to(device)
        ids = ids.to(device)

        features = model(crops)
        embeddings = torch.nn.functional.normalize(features)

        loss, valid_triplets = batch_hard_triplet_loss(embeddings, ids, margin)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        loss_train += loss.item()
        num_batches += 1

        if log_interval > 0 and idx_batch % log_interval == 0:
            wandb.log({
                "train/lr": scheduler.get_last_lr()[0],
                "train/loss": loss.item(),
                "train/valid_triplets": valid_triplets,
                "train/epoch": epoch
            })

    # Save the model at the end of each epoch
    root_ckpts.mkdir(exist_ok=True)
    torch.save(model.state_dict(), root_ckpts / f"{run_name}.pt")

    if num_batches == 0:
        return 0.0

    return loss_train / num_batches


def training_loop(
    num_epochs: int,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler,
    model: nn.Module,
    train_dataloader: torch.utils.data.DataLoader,
    device: torch.device,
) -> None:
    """Executes the training loop."""
    for epoch in tqdm(range(1, num_epochs + 1), "Epoch"):
        loss_train = train_epoch(
            model,
            train_dataloader,
            device,
            optimizer,
            scheduler,
            epoch
        )

        wandb.log({
            "train/avg_epoch_loss": loss_train,
            "train/epoch": epoch
        })

In [25]:
def train(model: nn.Module,
          lr: float,
          num_epochs: int,
          train_dataloader: torch.utils.data.DataLoader,
          device: torch.device,
          run_name: str,
          wandb_project: str,) -> None:
    """Executes the training loop."""
    # Logging
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("wandb_api_key") 
    wandb.login(key=wandb_key)
    wandb.init(
        project=wandb_project,
        name=run_name,
        settings=wandb.Settings(_disable_stats=True),
    )

    optimizer = torch.optim.SGD(
        [
            {"params": model.backbone.parameters()},
            {"params": model.emb.parameters(), "lr": 10 * lr},
            {"params": model.bnneck.parameters(), "lr": 10 * lr}
        ], lr=lr, momentum=0.9)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=len(train_dataloader)*num_epochs, eta_min=1e-6
    )
    
    training_loop(
        num_epochs,
        optimizer,
        scheduler,
        model,
        train_dataloader,
        device,
    )

    wandb.finish()

## Train Step

In [26]:
if do_train:
    train(
        embedder,
        lr,
        num_epochs,
        train_dataloader,
        device,
        run_name,
        wandb_project
    )
else:
    print("Training skip.")

Training skip.
